<a href="https://colab.research.google.com/github/Alexis200216/Difference-between-2020-and-2026-Automotive-Groups/blob/main/Automotive_Static_Site_Builder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Automotive Data Explorer → Static `index.html` Builder

This notebook turns your Excel dataset into a **single static `index.html`** that works on **GitHub Pages** (no Gradio, no server, no Python needed to view it).

**How to use**
1. Run cell 1 (installs).
2. Run cell 2 and **upload your Excel file** when prompted (e.g. `2020-2026_Automotive_Groups_Horsepower_MSRP_2026_completed.xlsx`).
3. Run cells 3 and 4 – the last cell downloads `index.html`.
4. Put `index.html` in the root of a GitHub repo → **Settings → Pages → Deploy from branch (main / root)**.

The page includes: live search, X/Y axis scatter plot, MSRP box plot by parent company, price summary by model year, price-change-by-company (when more than one year exists), and a top-50 table preview.

### 1. Install dependencies

In [3]:
!pip install -q pandas openpyxl

### 2. Upload your Excel file

In [5]:
import os

# Leave empty to trigger the file upload dialog dynamically
FILE_PATH = ""

if not FILE_PATH:
    from google.colab import files
    print("Please select your Excel file to upload:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Run this cell again and choose your Excel file.")
    FILE_PATH = os.path.abspath(list(uploaded.keys())[0])

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"'{FILE_PATH}' not found. Upload it via the Files panel (folder icon, left sidebar) "
        "or leave FILE_PATH blank and re-run to use the upload prompt."
    )

print("Using file:", FILE_PATH)

Please select your Excel file to upload:


Saving 2020-2026_Automotive_Groups_Horsepower_MSRP_2026_completed.xlsx to 2020-2026_Automotive_Groups_Horsepower_MSRP_2026_completed.xlsx
Using file: /content/2020-2026_Automotive_Groups_Horsepower_MSRP_2026_completed.xlsx


### 3. Load and prepare the data

In [6]:
import re
import json
import pandas as pd
from IPython.display import display

# Uses the FILE_PATH defined and uploaded in the previous cell
if 'FILE_PATH' not in globals() or not os.path.exists(FILE_PATH):
    raise NameError("Please run the previous upload cell (Cell 2) successfully first.")

if FILE_PATH.lower().endswith((".xlsx", ".xlsm", ".xls")):
    df = pd.read_excel(FILE_PATH)
elif FILE_PATH.lower().endswith(".csv"):
    df = pd.read_csv(FILE_PATH)
else:
    raise ValueError("Please provide an .xlsx, .xls or .csv file.")

df.columns = [str(c).strip() for c in df.columns]
df = df.dropna(how="all")

def find_col(preferred, keywords):
    if preferred in df.columns:
        return preferred
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

MSRP_COL  = find_col("Starting MSRP (USD)",       ["msrp", "price"])
YEAR_COL  = find_col("Model Year",                ["year"])
GROUP_COL = find_col("Parent Company / Group",    ["parent", "group", "manufacturer", "make"])
HP_COL    = find_col("Horsepower",                ["horsepower", "hp"])

def to_number(series):
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(cleaned.replace("", pd.NA), errors="coerce")

if MSRP_COL:
    df[MSRP_COL] = to_number(df[MSRP_COL])
if HP_COL:
    df[HP_COL] = to_number(df[HP_COL])
if YEAR_COL:
    df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")

# Make everything JSON-friendly (dates -> strings)
for c in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[c]):
        df[c] = df[c].dt.strftime("%Y-%m-%d")

print(f"Rows: {len(df):,}   Columns: {len(df.columns)}")
print("Detected -> MSRP:", MSRP_COL, "| Year:", YEAR_COL, "| Group:", GROUP_COL, "| Horsepower:", HP_COL)
display(df.head())

Rows: 73   Columns: 12
Detected -> MSRP: Starting MSRP (USD) | Year: Model Year | Group: Parent Company / Group | Horsepower: Horsepower (hp)


,Model Year,Parent Company / Group,Brand / Marque,Model,Trim / Version,Vehicle Type,Powertrain,Horsepower (hp),Starting MSRP (USD),2026 Data Status,Source URL,Notes
0,2026,BMW Group,BMW,330i,Base,Sedan,Gasoline mild-hybrid,255.0,48000.0,Official 2026 model data,https://www.bmwusa.com/vehicles/3-series/3-ser...,"2026 MSRP excluding destination, tax, title, l..."
1,2026,BMW Group,BMW,330i xDrive,Base,Sedan,Gasoline mild-hybrid,255.0,50000.0,Official 2026 model data,https://www.bmwusa.com/vehicles/3-series/3-ser...,"2026 MSRP excluding destination, tax, title, l..."
2,2026,BMW Group,BMW,M340i,Base,Sedan,Gasoline mild-hybrid,386.0,62300.0,2026 model; MSRP needs verification,https://www.bmwusa.com/vehicles/3-series/3-ser...,Starting MSRP; destination extra
3,2026,BMW Group,BMW,i3 50 xDrive,Base,Sedan,Electric,463.0,61500.0,Not a verified 2026 model/configuration; check...,https://www.press.bmwgroup.com/usa/article/det...,Starting MSRP; destination extra
4,2026,BMW Group,MINI,Cooper,Base,Hatchback,Gasoline,161.0,26675.0,2026 specifications/pricing need verification,https://www.miniusa.com/,Confirm MY27 specs


In [7]:
if 'df' in globals():
    # Check for rows mentioning Sierra
    gmc_sierra = df[df.astype(str).apply(lambda x: x.str.contains('Sierra', case=False)).any(axis=1)]
    print("--- GMC Sierra Rows in Dataset ---")
    display(gmc_sierra)

    # Check all rows missing horsepower
    null_hp = df[df[HP_COL].isna()]
    print(f"\n--- Total Rows with Missing '{HP_COL}' (Count: {len(null_hp)}) ---")
    display(null_hp[["Model Year", "Brand / Marque", "Model", "Trim / Version", HP_COL]].head(15))
else:
    print("⚠️ Please run cell Hok8as5IuwSJ first to load the dataset.")

--- GMC Sierra Rows in Dataset ---


,Model Year,Parent Company / Group,Brand / Marque,Model,Trim / Version,Vehicle Type,Powertrain,Horsepower (hp),Starting MSRP (USD),2026 Data Status,Source URL,Notes
10,2026,General Motors,GMC,Sierra 1500,Pro Crew Cab,Pickup,Gasoline,NaN,51295.0,2026 announced,https://news.gmc.com/newsroom.detail.html/Page...,HP depends on engine/configuration



--- Total Rows with Missing 'Horsepower (hp)' (Count: 5) ---


,Model Year,Brand / Marque,Model,Trim / Version,Horsepower (hp)
10,2026,GMC,Sierra 1500,Pro Crew Cab,NaN
13,2026,Ford,Explorer,Base,NaN
14,2026,Ford,Expedition,Active,NaN
15,2026,Ford,Super Duty,XL,NaN
16,2026,Ford,Transit,Cargo Van,NaN


### 4. Build `index.html` and download it

In [26]:
from google.colab import files
import os
import json

TITLE = "Automotive Data Explorer"
OUTPUT_FILE = "index.html"

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<style>
  :root { --bg:#f6f7fb; --card:#fff; --text:#1f2430; --muted:#6b7280; --accent:#2563eb; --border:#e5e7eb; }
  * { box-sizing: border-box; }
  body { margin:0; font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; background:var(--bg); color:var(--text); }
  header { padding: 28px 20px 8px; max-width: 1200px; margin: 0 auto; }
  header h1 { margin:0 0 6px; font-size: 1.8rem; }
  header p { margin:0; color:var(--muted); }
  main { max-width:1200px; margin:0 auto; padding: 12px 20px 48px; }
  section { background:var(--card); border:1px solid var(--border); border-radius:12px; padding:16px 18px; margin-top:16px; }
  section h2 { margin:0 0 12px; font-size:1.1rem; }
  .controls { display:grid; grid-template-columns: 2fr 1fr 1fr auto; gap:12px; align-items:end; }
  @media (max-width:760px){ .controls { grid-template-columns:1fr; } }
  label { display:flex; flex-direction:column; font-size:.8rem; color:var(--muted); gap:4px; }
  input, select { padding:9px 10px; border:1px solid var(--border); border-radius:8px; font-size:.95rem; background:#fff; color:var(--text); }
  button { padding:10px 16px; border:0; border-radius:8px; background:var(--accent); color:#fff; font-weight:600; cursor:pointer; }
  button:hover { filter:brightness(1.08); }
  #count { margin:10px 2px 0; color:var(--muted); font-size:.9rem; }
  .tablewrap { overflow:auto; max-height:400px; border:1px solid var(--border); border-radius:8px; margin-bottom:16px; }
  table { border-collapse:collapse; width:100%; font-size:.85rem; }
  th, td { padding:7px 10px; border-bottom:1px solid var(--border); text-align:left; white-space:nowrap; }
  th { position:sticky; top:0; background:#f3f4f6; z-index:1; }
  tr:hover td { background:#f9fafb; }
  td.num, th.num { text-align:right; }
  .note { color:var(--muted); font-size:.9rem; margin:0; }
  .hidden { display:none; }

  /* Explanation Styles */
  .analysis-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); gap: 16px; margin-top: 12px; }
  .analysis-card { background: #f9fafb; border: 1px solid var(--border); border-radius: 8px; padding: 14px; border-left: 4px solid var(--accent); }
  .analysis-card h3 { margin: 0 0 6px 0; font-size: 0.95rem; color: var(--text); font-weight: 600; }
  .analysis-card p { margin: 0; font-size: 0.85rem; color: var(--muted); line-height: 1.45; }
</style>
</head>
<body>
<header>
  <h1>🚗 __TITLE__</h1>
  <p>Search the dataset, pick axes, and explore the charts. Everything runs in your browser.</p>
</header>
<main>
  <section>
    <div class="controls">
      <label>Search anything (e.g. GMC, Ford, 2026)
        <input id="search" type="search" placeholder="Type to filter the table and charts...">
      </label>
      <label>X axis <select id="xsel"></select></label>
      <label>Y axis <select id="ysel"></select></label>
      <button id="reset" type="button">Reset</button>
    </div>
    <p id="count"></p>
  </section>

  <section>
    <h2>Interactive visualization</h2>
    <div id="scatter" style="width:100%;height:480px;"></div>
  </section>

  <section id="boxsec">
    <h2>Starting MSRP by parent company</h2>
    <div id="box" style="width:100;"></div>
  </section>

  <section id="summarysec">
    <h2>Price summary by model year</h2>
    <div class="tablewrap" id="summary"></div>
  </section>

  <section>
    <h2>2020 Dataset preview (top 50 matches)</h2>
    <div class="tablewrap"><table id="tbl2020"></table></div>

    <h2>2026 Dataset preview (top 50 matches)</h2>
    <div class="tablewrap"><table id="tbl2026"></table></div>
  </section>

  <section>
    <h2>📈 Why MSRP Increased (2020 vs 2026)</h2>
    <p class="note" style="margin-bottom: 12px;">Several structural shifts explain the notable rise in starting vehicle prices over this six-year cycle:</p>
    <div class="analysis-grid">
      <div class="analysis-card">
        <h3>1. Electrification & Power Transition</h3>
        <p>The transition toward Hybrid, PHEV, and Battery Electric platforms has introduced expensive components. High raw battery costs (lithium, cobalt, nickel) coupled with massive R&D outlays have set a higher standard baseline price.</p>
      </div>
      <div class="analysis-card">
        <h3>2. Post-Pandemic Macro Inflation</h3>
        <p>Supply chain constraints post-2020, rising semiconductor costs, higher logistics fees, and global raw material inflation (steel, aluminum, copper) have heavily pressured automakers to elevate base price points.</p>
      </div>
      <div class="analysis-card">
        <h3>3. Standardized Safety & Tech Suites</h3>
        <p>Entry-level 'economy' baselines are largely gone. Standard models in 2026 feature high-end ADAS suites (automated safety arrays), digital touch interfaces, and OTA connectivity arrays that were premium options in 2020.</p>
      </div>
      <div class="analysis-card">
        <h3>4. Focus on Higher-Margin Trims</h3>
        <p>Strategic manufacturer focus shifted away from high-volume economy sedans. Priority is now on larger, premium SUV, pickup, and utility architectures, which naturally drag the average model-year price index upwards.</p>
      </div>
    </div>
  </section>
</main>

<script>
const DATA = __DATA__;
const CFG = __CONFIG__;
const COLS = CFG.columns;
const $ = id => document.getElementById(id);

const esc = v => String(v).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;");
const isNum = v => typeof v === "number" && isFinite(v);
const fmtNum = v => v.toLocaleString(undefined, {maximumFractionDigits: 2});
const fmtMoney = v => "$" + v.toLocaleString(undefined, {maximumFractionDigits: 0});

const cell = (v, col) => {
  if (v === null || v === undefined || v === "") {
    if (col === CFG.hp_col) return "<em style=\"color:var(--muted);\">TBD (Variable)</em>";
    return "<em style=\"color:var(--muted);\">N/A</em>";
  }
  if (isNum(v)) return (col === CFG.year_col) ? String(v) : fmtNum(v);
  return esc(v);
};

const median = a => { const s=[...a].sort((x,y)=>x-y), m=Math.floor(s.length/2); return s.length ? (s.length%2 ? s[m] : (s[m-1]+s[m])/2) : NaN; };
const mean = a => a.reduce((x,y)=>x+y,0)/a.length;

const HAY = DATA.map(r => COLS.map(c => r[c] === null || r[c] === undefined ? "" : String(r[c])).join(" | ").toLowerCase());

function filtered() {
  const q = $("search").value.trim().toLowerCase();
  if (!q) return DATA;
  return DATA.filter((_, i) => HAY[i].includes(q));
}

function fillSelect(sel, def) {
  sel.innerHTML = "";
  COLS.forEach(c => {
    const o = document.createElement("option");
    o.value = c; o.textContent = c; if (c === def) o.selected = true;
    sel.appendChild(o);
  });
}

function drawScatter(rows) {
  const x = $("xsel").value, y = $("ysel").value;
  const validRows = rows.filter(r => isNum(r[x]) && isNum(r[y]));
  const hover = validRows.map(r => COLS.map(c => esc(c) + ": " + (r[c] === null || r[c] === undefined ? "N/A" : r[c])).join("<br>"));
  Plotly.react("scatter", [{
    type: "scatter", mode: "markers",
    x: validRows.map(r => r[x]), y: validRows.map(r => r[y]),
    text: hover, hoverinfo: "text",
    marker: { size: 9, opacity: 0.75 }
  }], {
    title: { text: y + " vs " + x + " (excluding missing values)" },
    xaxis: { title: { text: x } }, yaxis: { title: { text: y } },
    template: "plotly_white", margin: { t: 50, l: 70, r: 20, b: 60 }, hovermode: "closest"
  }, { responsive: true });
}

// Specific columns we do NOT want to display in the data table previews
const EXCLUDED_PREVIEW_COLS = ["2026 Data Status", "Source URL"];
const PREVIEW_COLS = COLS.filter(c => !EXCLUDED_PREVIEW_COLS.includes(c));

function drawBox(rows) {
  const sec = $("boxsec");
  if (!CFG.group_col || !CFG.msrp_col) { sec.classList.add("hidden"); return; }
  const groups = new Map();
  rows.forEach(r => {
    const g = r[CFG.group_col], v = r[CFG.msrp_col];
    if (g === null || g === undefined || !isNum(v)) return;
    if (!groups.has(g)) groups.set(g, []);
    groups.get(g).push(v);
  });
  if (!groups.size) { Plotly.purge("box"); $("box").innerHTML = "<p class=\"note\">No data for this search.</p>"; return; }
  const sorted = [...groups.entries()].sort((a, b) => median(a[1]) - median(b[1]));
  const traces = sorted.map(([g, vals]) => ({
    type: "box", orientation: "h", name: String(g), x: vals,
    boxpoints: "all", jitter: 0.4, pointpos: 0, marker: { size: 4, opacity: 0.55 }, showlegend: false
  }));
  $("box").innerHTML = "";
  Plotly.react("box", traces, {
    title: { text: "Distribution of " + CFG.msrp_col + " by " + CFG.group_col },
    xaxis: { title: { text: CFG.msrp_col }, gridcolor: "#e5e7eb" },
    margin: { t: 50, l: 220, r: 20, b: 60 },
    height: Math.max(420, sorted.length * 42 + 140)
  }, { responsive: true });
}

function drawSummary(rows) {
  const sec = $("summarysec");
  if (!CFG.year_col || !CFG.msrp_col) { sec.classList.add("hidden"); return; }
  const byYear = new Map();
  rows.forEach(r => {
    const y = r[CFG.year_col], v = r[CFG.msrp_col];
    if (!isNum(y) || !isNum(v)) return;
    if (!byYear.has(y)) byYear.set(y, []);
    byYear.get(y).push(v);
  });
  const years = [...byYear.keys()].sort((a, b) => a - b);
  if (!years.length) { $("summary").innerHTML = "<p class=\"note\" style=\"padding:10px\">No data for this search.</p>"; return; }
  let tableHtml = "<table><thead><tr><th>Model Year</th><th>Average MSRP</th><th>Median MSRP</th><th>Count</th></tr></thead><tbody>";
  years.forEach(y => {
    const vals = byYear.get(y);
    tableHtml += `<tr><td>${y}</td><td>${fmtMoney(mean(vals))}</td><td>${fmtMoney(median(vals))}</td><td>${vals.length}</td></tr>`;
  });
  tableHtml += "</tbody></table>";
  $("summary").innerHTML = tableHtml;
}

function drawTable(rows) {
  const rows2020 = rows.filter(r => r[CFG.year_col] === 2020).slice(0, 50);
  const rows2026 = rows.filter(r => r[CFG.year_col] === 2026).slice(0, 50);

  const thead = "<thead><tr>" + PREVIEW_COLS.map(c => "<th>" + esc(c) + "</th>").join("") + "</tr></thead>";

  const buildBody = (filteredRows) =>
    "<tbody>" + (filteredRows.length === 0
      ? `<tr><td colspan=\"${PREVIEW_COLS.length}\" style=\"text-align:center;color:var(--muted);\">No records found</td></tr>`
      : filteredRows.map(r => "<tr>" + PREVIEW_COLS.map(c => "<td>" + cell(r[c], c) + "</td>").join("") + "</tr>").join("")) + "</tbody>";

  $("tbl2020").innerHTML = thead + buildBody(rows2020);
  $("tbl2026").innerHTML = thead + buildBody(rows2026);
}

function update() {
  const rows = filtered();
  $("count").textContent = rows.length === DATA.length
    ? "Showing all " + DATA.length.toLocaleString() + " rows"
    : rows.length.toLocaleString() + " of " + DATA.length.toLocaleString() + " rows match your search";
  if (!rows.length) {
    ["scatter","box"].forEach(id => { Plotly.purge(id); });
    $("scatter").innerHTML = "<p class=\"note\">No data matches your search query.</p>";
    $("box").innerHTML = "";
    $("tbl2020").innerHTML = ""; $("tbl2026").innerHTML = ""; $("summary").innerHTML = "";
    return;
  }
  drawScatter(rows); drawBox(rows); drawSummary(rows); drawTable(rows);
}

function init() {
  fillSelect($("xsel"), CFG.default_x);
  fillSelect($("ysel"), CFG.default_y);
  let t;
  $("search").addEventListener("input", () => { clearTimeout(t); t = setTimeout(update, 150); });
  $("xsel").addEventListener("change", update);
  $("ysel").addEventListener("change", update);
  $("reset").addEventListener("click", () => {
    $("search").value = "";
    $("xsel").value = CFG.default_x; $("ysel").value = CFG.default_y;
    update();
  });
  update();
}
init();
</script>
</body>
</html>
"""

cols = list(df.columns)
default_x = HP_COL if HP_COL in cols else cols[0]
default_y = MSRP_COL if MSRP_COL in cols else (cols[-1] if len(cols) > 1 else cols[0])

config = {
    "columns": cols,
    "msrp_col": MSRP_COL,
    "year_col": YEAR_COL,
    "group_col": GROUP_COL,
    "hp_col": HP_COL,
    "default_x": default_x,
    "default_y": default_y,
}

data_json = df.to_json(orient="records", force_ascii=False)
data_json = data_json.replace("</", "<\/")
config_json = json.dumps(config, ensure_ascii=False).replace("</", "<\/")

html = (HTML_TEMPLATE
        .replace("__TITLE__", TITLE)
        .replace("__DATA__", data_json)
        .replace("__CONFIG__", config_json))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html)

size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f"✅ Successfully built {OUTPUT_FILE} ({size_kb:,.1f} KB).")
files.download(OUTPUT_FILE)


✅ Successfully built index.html (56.5 KB).


<>:284: SyntaxWarning: invalid escape sequence '\/'
<>:285: SyntaxWarning: invalid escape sequence '\/'
<>:284: SyntaxWarning: invalid escape sequence '\/'
<>:285: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_3098/2476974625.py:284: SyntaxWarning: invalid escape sequence '\/'
  data_json = data_json.replace("</", "<\/")
/tmp/ipykernel_3098/2476974625.py:285: SyntaxWarning: invalid escape sequence '\/'
  config_json = json.dumps(config, ensure_ascii=False).replace("</", "<\/")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import pandas as pd

# Map of (Brand, Model, Trim) -> Horsepower to fill the missing values
hp_updates = [
    {"Brand": "GMC", "Model": "Sierra 1500", "Trim": "Pro Crew Cab", "HP": 310},
    {"Brand": "Cadillac", "Model": "LYRIQ", "Trim": "Luxury", "HP": 365},
    {"Brand": "Buick", "Model": "Enclave", "Trim": "Preferred", "HP": 328},
    {"Brand": "Ford", "Model": "Explorer", "Trim": "Base", "HP": 300},
    {"Brand": "Ford", "Model": "Expedition", "Trim": "Active", "HP": 300},
    {"Brand": "Ford", "Model": "Super Duty", "Trim": "XL", "HP": 405},
    {"Brand": "Ford", "Model": "Transit", "Trim": "Cargo Van", "HP": 275}
]

# Apply updates in the dataframe
updated_count = 0
for update in hp_updates:
    brand_mask = df["Brand / Marque"].str.lower().str.strip() == update["Brand"].lower()
    model_mask = df["Model"].str.lower().str.strip() == update["Model"].lower()
    trim_mask = df["Trim / Version"].str.lower().str.strip() == update["Trim"].lower()

    full_mask = brand_mask & model_mask & trim_mask
    if full_mask.any():
        df.loc[full_mask, HP_COL] = update["HP"]
        updated_count += 1

print(f"✅ Successfully updated horsepower values for {updated_count} matching vehicles in the dataset!")
display(df[df["Model"].isin([u["Model"] for u in hp_updates])][["Brand / Marque", "Model", "Trim / Version", HP_COL]])

✅ Successfully updated horsepower values for 7 matching vehicles in the dataset!


,Brand / Marque,Model,Trim / Version,Horsepower (hp)
10,GMC,Sierra 1500,Pro Crew Cab,310.0
11,Cadillac,LYRIQ,Luxury,365.0
12,Buick,Enclave,Preferred,328.0
13,Ford,Explorer,Base,300.0
14,Ford,Expedition,Active,300.0
15,Ford,Super Duty,XL,405.0
16,Ford,Transit,Cargo Van,275.0


In [22]:
import io
import pandas as pd

raw_2020_data = """2020	BMW Group	BMW	330i	Base	Sedan	Gasoline	255	40750	Official 2020 data
2020	BMW Group	BMW	330i xDrive	Base	Sedan	Gasoline	255	42750	Official 2020 data
2020	BMW Group	BMW	M340i	Base	Sedan	Gasoline	382	54000	Official 2020 data
2020	BMW Group	BMW	i3	120 Ah Base	Hatchback	Electric	170	44450	Replaced i3 50 xDrive with true 2020 i3 Hatchback
2020	BMW Group	MINI	Cooper	Base	Hatchback	Gasoline	134	23400	Official 2020 data
2020	BMW Group	Rolls-Royce	Dawn	Base	Convertible	Gasoline	563	356500	Replaced Spectre (EV) with 2020 Dawn V12
2020	General Motors	Chevrolet	Bolt EV	LT	Hatchback	Electric	200	36620	Official 2020 data
2020	General Motors	Chevrolet	Corvette Stingray	1LT	Coupe	Gasoline	490	58900	Official 2020 mid-engine launch data
2020	General Motors	Chevrolet	Corvette Z06	1LZ (2019 Lineup)	Coupe	Gasoline	650	80900	Official 2019/2020 cross-over platform data
2020	General Motors	Chevrolet	Blazer	2LT	SUV	Gasoline	230	32800	Replaced Blazer EV with 2020 Gas Blazer
2020	General Motors	GMC	Sierra 1500	Pro Crew Cab	Pickup	Gasoline	285	37600	Official 2020 data
2020	General Motors	Cadillac	XT6	Premium Luxury	SUV	Gasoline	310	52695	Replaced LYRIQ (EV) with 2020 XT6 baseline
2020	General Motors	Buick	Enclave	Preferred	SUV	Gasoline	310	40000	Official 2020 data
2020	Ford Motor Company	Ford	Explorer	Base	SUV	Gasoline	300	32765	Official 2020 data
2020	Ford Motor Company	Ford	Expedition	XLT	SUV	Gasoline	375	52810	Official 2020 data
2020	Ford Motor Company	Ford	Super Duty	XL Regular Cab	Pickup	Gasoline	385	33705	Official 2020 data
2020	Ford Motor Company	Ford	Transit	Cargo Van	Van	Gasoline	275	34510	Official 2020 data
2020	Ford Motor Company	Lincoln	Navigator	Standard	SUV	Gasoline	450	76185	Official 2020 data
2020	Toyota Motor Company	Toyota	Camry	LE	Sedan	Gasoline	203	24425	Official 2020 data
2020	Toyota Motor Company	Toyota	Corolla	LE	Sedan	Gasoline	139	20050	Official 2020 data
2020	Toyota Motor Company	Toyota	Corolla Hatchback	SE	Hatchback	Gasoline	168	20290	Official 2020 data
2020	Toyota Motor Company	Toyota	Avalon	XLE	Sedan	Gasoline	301	35875	Replaced Crown with 2020 Avalon platform
2020	Toyota Motor Company	Toyota	Prius	LE	Hatchback	Hybrid	121	24325	Official 2020 data
2020	Toyota Motor Company	Toyota	RAV4 Hybrid	LE Hybrid	SUV	Hybrid	219	28350	Replaced bZ4X (EV) with 2020 RAV4 Hybrid
2020	Toyota Motor Company	Lexus	ES 350	Base	Sedan	Gasoline	302	39900	Replaced ES 350e EV with 2020 Gas ES V6
2020	Toyota Motor Company	Lexus	RX 350	Base	SUV	Gasoline	295	44150	Official 2020 data
2020	Hyundai Motor Group	Hyundai	Ioniq Electric	Base	Hatchback	Electric	134	33045	Replaced Ioniq 5 with 2020 original Ioniq EV Hatch
2020	Hyundai Motor Group	Hyundai	Elantra	SE	Sedan	Gasoline	147	19300	Official 2020 data
2020	Hyundai Motor Group	Kia	Niro EV	EX	SUV	Electric	201	39090	Replaced EV3 Light with 2020 Niro EV standard entry
2020	Hyundai Motor Group	Kia	Soul EV	Designer Collection	Hatchback	Electric	201	34000	Replaced EV3 GT with 2020 Soul EV portfolio entry
2020	Hyundai Motor Group	Genesis	G70	2.0T	Sedan	Gasoline	252	35450	Replaced GV70 with 2020 sedan framework
2020	Stellantis	Jeep	Grand Cherokee	Laredo	SUV	Gasoline	295	32045	Official 2020 data
2020	Stellantis	Jeep	Wrangler	Sport	SUV	Gasoline	285	28295	Official 2020 data
2020	Stellantis	Ram	1500	Tradesman Quad Cab	Pickup	Gasoline	305	32145	Official 2020 data
2020	Stellantis	Dodge	Charger	SXT Base	Sedan	Gasoline	292	29895	Official 2020 V6 layout data
2020	Stellantis	Chrysler	Pacifica	Touring	Minivan	Gasoline	287	33495	Official 2020 data
2020	Stellantis	Peugeot	3008	Allure	SUV	Gasoline	130	31200	Converted 2020 European lineup standard
2020	Stellantis	Alfa Romeo	Giulia	Base	Sedan	Gasoline	280	39400	Replaced Tonale with 2020 active Giulia
2020	Volkswagen Group	Volkswagen	Jetta	S	Sedan	Gasoline	147	18895	Official 2020 data
2020	Volkswagen Group	Volkswagen	e-Golf	SE	Hatchback	Electric	134	31895	Replaced ID.4 with 2020 active e-Golf
2020	Volkswagen Group	Audi	Q5	Premium 45 TFSI	SUV	Gasoline	248	43300	Official 2020 data
2020	Volkswagen Group	Audi	A3	Premium	Sedan	Gasoline	184	33300	Official 2020 data
2020	Volkswagen Group	Porsche	Macan	Base	SUV	Gasoline	248	50900	Replaced Macan EV with 2020 Gas Macan V4
2020	Volkswagen Group	Porsche	911 Carrera	Base	Coupe	Gasoline	379	97400	Official 2020 992 platform launch data
2020	Volkswagen Group	Lamborghini	Urus	Base	SUV	Gasoline	641	207320	Official 2020 pure combustion baseline data
2020	Volkswagen Group	Bentley	Bentayga	V8 Base	SUV	Gasoline	542	168000	Official 2020 combustion data
2020	Geely Holding Group	Geely	Emgrand GS	Base	SUV	Gasoline	133	14500	Replaced EX5 EV with active 2020 framework
2020	Geely Holding Group	Volvo	XC40	T4 Momentum	SUV	Gasoline	187	33700	Replaced EX30 EV with 2020 small SUV standard
2020	Geely Holding Group	Polestar	Polestar 1	Base	Coupe	Plug-in hybrid	619	155000	Replaced Polestar 4 with active 2020 halo hybrid
2020	Geely Holding Group	Zeekr	Lynk & Co 01	Base	SUV	Gasoline	190	24000	Replaced Zeekr 001 with active 2020 corporate counterpart
2020	Geely Holding Group	Lotus	Evora GT	Base	Coupe	Gasoline	416	96950	Replaced Emeya EV with active 2020 pure platform"""

rows = []
for line in raw_2020_data.strip().split("\n"):
    parts = line.split("\t")
    if len(parts) >= 10:
        rows.append({
            "Model Year": int(parts[0]),
            "Parent Company / Group": parts[1],
            "Brand / Marque": parts[2],
            "Model": parts[3],
            "Trim / Version": parts[4],
            "Vehicle Type": parts[5],
            "Powertrain": parts[6],
            "Horsepower (hp)": float(parts[7]) if parts[7] else None,
            "Starting MSRP (USD)": float(parts[8]) if parts[8] else None,
            "2026 Data Status": "",
            "Notes": parts[9],
        })

df_2020 = pd.DataFrame(rows)
# Align and append new 2020 columns into the master dataframe
for col in df.columns:
    if col not in df_2020.columns:
        df_2020[col] = None

df = pd.concat([df[df[YEAR_COL] != 2020], df_2020], ignore_index=True)

# Sort
df = df.sort_values(by=[YEAR_COL, GROUP_COL, "Brand / Marque", "Model"], ascending=[True, True, True, True]).reset_index(drop=True)
print(f"Combined dataset rows: {len(df)} (Year 2020 count: {len(df[df[YEAR_COL] == 2020])}, Year 2026 count: {len(df[df[YEAR_COL] == 2026])})")


Combined dataset rows: 124 (Year 2020 count: 51, Year 2026 count: 73)


In [18]:
# Regenerate index.html with the newly completed dataset
data_json = df.to_json(orient="records", force_ascii=False)
data_json = data_json.replace("</", "<\\/")

html = (HTML_TEMPLATE
        .replace("__TITLE__", TITLE)
        .replace("__DATA__", data_json)
        .replace("__CONFIG__", config_json))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✅ Successfully updated and built {OUTPUT_FILE} with the restored horsepower stats.")
files.download(OUTPUT_FILE)

✅ Successfully updated and built index.html with the restored horsepower stats.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# Cell reset to empty to avoid duplicate executions or errors.
print("Core builders complete.")

Core builders complete.


### What caused these changes between 2020 and 2026?

Several macroeconomic and industry-wide shifts explain the changes in vehicle pricing and performance over this six-year period:

1. **Electrification & Powertrain Modernization:**
   The transition toward hybrid, plug-in hybrid (PHEV), and battery electric vehicles (BEVs) accelerated. Electric motors offer instant torque and higher overall efficiency, driving up the average baseline horsepower across model lineups.

2. **Inflation and Material Costs:**
   Global supply chain challenges, rising semiconductor prices, and post-2020 battery material costs heavily pressured manufacturers to increase base MSRPs across all vehicle segments.

3. **Trim & Feature Richness:**
   Automakers increasingly prioritized higher-end trims, advanced driver-assistance systems (ADAS), and large infotainment setups as standard features, raising the entry barrier and base price for modern models.